In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# <a href="https://en.wikipedia.org/wiki/Unification_(computer_science)">Unification</a>

This notebook implements the algorithm of *Martelli and Montanari* for the unification of terms.

## Utility Functions

Terms are represented as values of the type `Term` that is defined in the notebook `FOL-Parser.ipynb`:
```
type Term =
    | Var of string                  // variable
    | Fun of string * Term list      // function application, constants have no arguments
```
The notebook also provides the function `parseTerm` that converts a string into a term.  The parser distinguishes variables and function symbols as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
importNotebook "FOL-Parser.ipynb"

imported 

FOL-Parser.ipynb

In [3]:
parseTerm "F(G(x),y)"

Fun ("F", [Fun ("G", [Var "x"]); Var "y"])

A *substitution* $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$ is represented as a map from variables to terms.

The function `applyTerm` takes two arguments:
- `σ` is a substitution,
- `t` is a term.

It computes the term $t\sigma$, i.e. it *applies* the substitution $\sigma$ to $t$.  Assume that $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$.  Then $t\sigma$ is defined by induction on $t$:
- If $t$ is a variable, there are two cases when defining $t\sigma$:
  - $t = x_i$ for an $i\in\{1,\cdots,n\}$.  Then we define
    $$ x_i\sigma := t_i. $$
  - $t = y$ where $y\in\mathcal{V}$, but $y \not\in \{x_1,\cdots,x_n\}$. Then we define
    $$ y\sigma := y.$$
- Otherwise, we must have $t = f(s_1,\cdots,s_m)$. Then we define:
  $$ f(s_1, \cdots, s_m)\sigma := f(s_1\sigma, \cdots, s_m\sigma). $$

In [4]:
let rec applyTerm (σ: Map<string, Term>) (t: Term) : Term =
    match t with
    | Var x       -> match σ.TryFind x with
                     | Some s -> s
                     | None   -> t
    | Fun (f, ts) -> Fun (f, List.map (applyTerm σ) ts)

A *syntactical equation* $s \doteq t$ is represented as the pair `(s, t)`.  The function `applyEquations` takes two arguments:
- `σ` is a substitution,
- `E` is a set of syntactical equations.

It applies `σ` to both sides of all equations in `E`.

In [5]:
let applyEquations (σ: Map<string, Term>) (E: Set<Term * Term>) : Set<Term * Term> =
    Set.map (fun (s, t) -> (applyTerm σ s, applyTerm σ t)) E

In [6]:
let s1 = parseTerm "G(z)"
let s2 = parseTerm "H(u, v)"
let σ  = Map [ ("x", s1); ("y", s2) ]
σ

map [("x", Fun ("G", [Var "z"])); ("y", Fun ("H", [Var "u"; Var "v"]))]

In [7]:
let t = parseTerm "F(x,H(y,x),G(z))"
termToString t

"F(x, H(y, x), G(z))"

In [8]:
termToString (applyTerm σ t)

"F(G(z), H(H(u, v), G(z)), G(z))"

If  $\sigma = \big\{ x_1 \mapsto s_1, \cdots, x_m \mapsto s_m \big\}$ and
$\tau = \big\{ y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$
are two substitutions that are <em style="color:blue;">non-overlapping</em>, i.e. such that $\texttt{dom}(\sigma) \cap \texttt{dom}(\tau) = \{\}$ holds,
then we define the <em style="color:blue;">composition</em> $\sigma\tau$ of $\sigma$ and $\tau$ as follows:
$$\sigma\tau := \big\{ x_1 \mapsto s_1\tau, \cdots, x_m \mapsto s_m\tau,\; y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$$
The function `compose` takes two arguments:
- `σ` is a substitution,
- `τ` is a substitution such that `σ` and `τ` are non-overlapping.

It returns the composition $\sigma\tau$.

**Implementation:** `Map.map` applies a function to all entries of a map; the function receives both the key and the value.  `Map.fold` is used to add all entries of `τ` to the resulting map.

In [9]:
let compose (σ: Map<string, Term>) (τ: Map<string, Term>) : Map<string, Term> =
    let στ = Map.map (fun _ s -> applyTerm τ s) σ
    Map.fold (fun (acc: Map<string, Term>) x t -> acc.Add(x, t)) στ τ

In [10]:
let τ = Map [ ("z", s1); ("u", s2) ]
τ

map [("u", Fun ("H", [Var "u"; Var "v"])); ("z", Fun ("G", [Var "z"]))]

In [11]:
compose σ τ

map [("u", Fun ("H", [Var "u"; Var "v"])); ("x", Fun ("G", [Fun ("G", [Var "z"])])); ("y", Fun ("H", [Fun ("H", [Var "u"; Var "v"]); Var "v"])); ("z", Fun ("G", [Var "z"]))]

The function `occurs` takes two arguments:
- `x` is a variable,
- `t` is a term.

It checks whether the variable $x$ occurs in the term $t$.

In [12]:
let rec occurs (x: string) (t: Term) : bool =
    match t with
    | Var y       -> x = y
    | Fun (_, ts) -> List.exists (occurs x) ts

In [13]:
occurs "u" t

false

In [14]:
occurs "x" t

true

## The Algorithm of Martelli and Montanari

The rules of Martelli and Montanari that can be used to solve a system of syntactical equations are as follows:
<ol>
<li> If $y\in\mathcal{V}$ is a variable that does <b style="color:red;">not</b> occur in the term $t$,
     then we perform the following reduction:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E\{y \mapsto t\}, \sigma\big\{ y \mapsto t \big\} \Big\rangle
     $$
</li>
<li> If the variable $y$ occurs in the term $t$, then the system of syntactical equations
     $E \cup \big\{ y \doteq t \big\}$ is not solvable:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle\;\leadsto\; \texttt{None} \quad
        \mbox{if $y \in \textrm{Var}(t)$ and $y \not=t$.}$$
</li>
<li> If $y\in\mathcal{V}$ is a variable and $t$ is no variable, then we use the following rule:
     $$ \Big\langle E \cup \big\{ t \doteq y \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle.
     $$
</li>
<li> Trivial syntactical equations of variables can be dropped:
     $$ \Big\langle E \cup \big\{ x \doteq x \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E, \sigma \Big\rangle.
     $$
</li>
<li> If $f$ is an $n$-ary function symbol, then we have:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_n) \doteq f(t_1,\cdots,t_n) \big\}, \sigma \Big\rangle
         \;\leadsto\;
         \Big\langle E \cup \big\{ s_1 \doteq t_1, \cdots, s_n \doteq t_n\}, \sigma \Big\rangle.
     $$
</li>
<li> The system of syntactical equations $E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\}$
     has <b style="color:red;">no</b> solution if the function symbols $f$ and $g$ are different:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\},
      \sigma \Big\rangle \;\leadsto\; \texttt{None} \qquad \mbox{if $f \not= g$}.
     $$
</ol>

The function `solve` takes two arguments:
- `E` is a set of <em style="color:blue;">syntactical equations</em>,
- `σ` is a substitution.

It uses the rules of Martelli and Montanari to solve $E$.  The result has the type `Map<string, Term> option`: if `E` is solvable, the result is `Some μ`, where `μ` is the composition of `σ` with the most general unifier of `E`.  Otherwise, `None` is returned.

**Implementation:** Every recursive call of `solve` performs one step of the algorithm.  We take an equation $s \doteq t$ from $E$ and apply the rule that matches this equation.  The function `List.zip` combines the arguments of the two function applications into a list of pairs, i.e. into new equations.

In [15]:
let rec solve (E: Set<Term * Term>) (σ: Map<string, Term>) : Map<string, Term> option =
    if E.IsEmpty then
        Some σ
    else
        let (s, t) = Set.minElement E
        let E = E.Remove (s, t)
        match s, t with
        | _ when s = t ->                     // remove trivial equations
            solve E σ
        | Var x, _ ->                         // s is a variable
            if occurs x t then
                None
            else
                let ρ = Map [ (x, t) ]
                solve (applyEquations ρ E) (compose σ ρ)
        | _, Var _ ->                         // t is a variable, but s is not
            solve (E.Add (t, s)) σ
        | Fun (f, sArgs), Fun (g, tArgs) ->
            if f <> g || sArgs.Length <> tArgs.Length then
                None
            else
                solve (E + set (List.zip sArgs tArgs)) σ

The function `unify` takes two arguments:
- `s` is a term,
- `t` is a term.

It returns the <em style="color:blue;">most general unifier</em> of $s$ and $t$ as `Some μ`, or `None` if $s$ and $t$ are not unifiable.

In [16]:
let unify (s: Term) (t: Term) : Map<string, Term> option =
    solve (set [ (s, t) ]) Map.empty

In [17]:
let t1 = parseTerm "P(x1,F(x1))"
let t2 = parseTerm "P(G(x2),x3)"
(termToString t1, termToString t2)

("P(x1, F(x1))", "P(G(x2), x3)")

In [18]:
let μ = unify t1 t2
μ

Some map [("x1", Fun ("G", [Var "x2"])); ("x3", Fun ("F", [Fun ("G", [Var "x2"])]))]

Let us check that $\mu$ is indeed a unifier, i.e. that $t_1\mu = t_2\mu$:

In [19]:
termToString (applyTerm μ.Value t1), termToString (applyTerm μ.Value t2)

("P(G(x2), F(G(x2)))", "P(G(x2), F(G(x2)))")

The terms $x$ and $F(x)$ are not unifiable:

In [20]:
unify (parseTerm "x") (parseTerm "F(x)")

None